# Final Test Results

In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda3/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('data_classifier.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [3]:
with open("scaffold_splits_clf.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=528  test=115  folds=5
Device: cpu


In [4]:
X_fp_RU_pooled = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly_pooled      = np.load("Vectors/X_fp_pooled_poly.npy")

y=np.load("Vectors/y.npy")

In [5]:
import importlib
import scripts.test_eval_lim_models_cls
importlib.reload(scripts.test_eval_lim_models_cls)

<module 'scripts.test_eval_lim_models_cls' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML/Section_S3/Classifier/scripts/test_eval_lim_models_cls.py'>

In [7]:
from scripts.test_eval_lim_models_cls import run_all_test

test_results = run_all_test(
    X_fp_RU_pooled, X_fp_poly_pooled,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_full.json",
)

Loaded HPs from best_hp_full.json  (21 entries)

── XGB (FP RU) ──────────────────────────────────────────

  TEST RESULTS: XGB (FP RU)
  Accuracy              : 0.6696
  Macro F1              : 0.4837
  AUROC                 : 0.6701
  AUPRC                 : 0.7869
  No Tm Precision       : 1.0000
  No Tm Recall          : 0.0952
  Has Tm Precision      : 0.6577
  Has Tm Recall         : 1.0000
  train_time_min        : 0.0100

  Subgroup: copolymer_type
  Group                    n      Acc       F1
  ---------------------------------------------
  alternating             12   0.5833   0.7368
  block                    5   0.8000   0.8889
  random                   6   0.6667   0.8000

  Subgroup: stereo_class
  Group                    n      Acc       F1
  ---------------------------------------------
  achiral                 10   0.6000   0.7500
  atactic                 28   0.2857   0.3750
  isotactic               36   0.8889   0.9412
  stereo-irregular        15   0.6000   0

In [8]:
from scripts.fig1_split_validation import plot_split_validation

In [10]:
plot_split_validation(df, trainval_idx, test_idx)

Saved: Figures/fig1c_split_validation-2-scaffold.pdf
